# 01 Explore: PaySim mobile money transactions

**Project:** a fraud review queue for a South African mobile money provider.
**This notebook:** loads the data and documents every column (Step 2), then measures the imbalance, finds where fraud happens and checks for leakage (Step 3).

**Data:** PaySim, a simulated mobile money log built from one month of real transactions at a mobile money service in an African country. Lopez-Rojas, Elmir and Axelsson (2016), *PaySim: A financial mobile money simulator for fraud detection*. The generated dataset is published on [Kaggle](https://www.kaggle.com/datasets/ealaxi/paysim1) under CC BY-SA 4.0; this project downloads an identical copy from a [Hugging Face mirror](https://huggingface.co/datasets/purulalwani/Synthetic-Financial-Datasets-For-Fraud-Detection) because it needs no login.

## 1. Load the data

In [1]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np

RAW = Path("../data/raw/paysim.parquet")
URL = ("https://huggingface.co/datasets/purulalwani/Synthetic-Financial-Datasets-For-Fraud-Detection"
       "/resolve/main/financial_txns.parquet")

if not RAW.exists():
    print("Downloading PaySim (about 270 MB) ...")
    urllib.request.urlretrieve(URL, RAW)

tx = pd.read_parquet(RAW)
print(f"{len(tx):,} transactions, {tx.shape[1]} columns")
tx.head()

6,362,620 transactions, 11 columns


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [2]:
tx.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 706.2 MB


In [3]:
tx.describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
step,6362620.0,243.4,142.3,1.0,156.0,239.0,335.0,743.0
amount,6362620.0,179861.9,603858.2,0.0,13389.6,74871.9,208721.5,92445516.6
oldbalanceOrg,6362620.0,833883.1,2888242.7,0.0,0.0,14208.0,107315.2,59585040.4
newbalanceOrig,6362620.0,855113.7,2924048.5,0.0,0.0,0.0,144258.4,49585040.4
oldbalanceDest,6362620.0,1100701.7,3399180.1,0.0,0.0,132705.7,943036.7,356015889.4
newbalanceDest,6362620.0,1224996.4,3674128.9,0.0,0.0,214661.4,1111909.2,356179278.9
isFraud,6362620.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
isFlaggedFraud,6362620.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 2. Checking what the columns mean

In [4]:
print("Transaction types:")
print(tx["type"].value_counts().to_string())
print(f"\nSteps run from {tx.step.min()} to {tx.step.max()} (one step = one hour, so {tx.step.max() / 24:.0f} days)")
print("\nFirst letter of account names:")
print("  origin:     ", tx.nameOrig.str[0].value_counts().to_dict())
print("  destination:", tx.nameDest.str[0].value_counts().to_dict())

Transaction types:
type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432

Steps run from 1 to 743 (one step = one hour, so 31 days)

First letter of account names:


  origin:      {'C': 6362620}


  destination: {'C': 4211125, 'M': 2151495}


In [5]:
merchant_dest = tx.nameDest.str.startswith("M")
print("Destination is a merchant (M...) for these types:", tx.loc[merchant_dest, "type"].unique().tolist())
print("Merchant destination balances are always zero:",
      bool((tx.loc[merchant_dest, ["oldbalanceDest", "newbalanceDest"]] == 0).all().all()))
print(f"\nisFlaggedFraud: {tx.isFlaggedFraud.sum()} transactions flagged by the provider's existing rule")

Destination is a merchant (M...) for these types: ['PAYMENT']


Merchant destination balances are always zero: True

isFlaggedFraud: 16 transactions flagged by the provider's existing rule


What this tells me for the data dictionary:

- `step` is the hour of the simulation (1 to 743, about 31 days). Hour of day and day number can be derived from it.
- Account names starting with **C** are customers, **M** are merchants. Merchants only receive `PAYMENT`s, and the simulator does not record merchant balances (always zero), so destination balances only mean something for customer-to-customer money.
- `isFlaggedFraud` is the provider's existing rule (it targets very large transfers). It fired only 16 times in a month, which is a useful "what the business does today" comparison in Step 4.
- `isFraud` is the label: the simulator marks transactions made by its fraudster agents, who take control of a customer's account and try to empty it by transferring the money out and cashing it out.